# Day 2 — Incident diagnosis (Jupyter + CMD)

**Prerequisite:** Day 1 complete (`start-lab.bat` done).

Today something looks **broken** on a ticket. You collect evidence in order:

1. **Consumer group** — lag and offsets  
2. **Application logs** — what the producer/consumer app saw  
3. **Broker health** — cluster Active, ISR, DNS  
4. **Recover** — restart consume path and produce a test line  

Each code cell = **CMD** (same as [commands.md](commands.md)). Concepts: [notes.md](notes.md).


## Setup — load lab session

Same as Day 1: every CMD window starts with `call set-kafka-lab.bat`. That sets `TOPIC`, `GROUP`, `BOOTSTRAP`, `CLIENT`, `CLUSTER_ARN`.

**log4j WARN** lines on Kafka commands are normal on Windows — ignore them.

**Two data sources today:**

| Source | Topic / group in output |
|--------|-------------------------|
| **Live cluster** (sections 1, 3, 4) | your `orders-userN`, `cg-userN-support` |
| **Sample logs** (section 2) | fictional `orders-lab`, `cg-lab-support` — for the ticket story only |


In [1]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo TOPIC=%TOPIC% GROUP=%GROUP% BOOTSTRAP=%BOOTSTRAP%

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>echo TOPIC=%TOPIC% GROUP=%GROUP% BOOTSTRAP=%BOOTSTRAP%
TOPIC=orders-user15 GROUP=cg-user15-support BOOTSTRAP=b-1-public.mskkafkaclass.qau5zr.c4.kafka.ap-south-1.amazonaws.com:9196

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

## 1 — Diagnose consumer lag

### Concept: LAG

For each **partition** your group reads:

| Column | Meaning |
|--------|--------|
| **CURRENT-OFFSET** | Where your group last saved progress (“I read up to here”) |
| **LOG-END-OFFSET** | Newest message position on the broker |
| **LAG** | Messages waiting = LOG-END minus CURRENT |

- **LAG = 0** — caught up  
- **LAG growing** — consumer stopped or slower than producers  
- **No active members** — no consumer running right now (common after you stop `consume.bat`)

**Command:** `kafka-consumer-groups.bat ... --group %GROUP% --describe`

**What healthy output looks like in the lab:**

- **LAG = 0** on partitions that have traffic — the group is caught up.
- **No active members** is normal when no consumer is running (for example after you stop `consume.bat`).
- **Offsets differ per partition** (partition 1 at `5`, partition 0 at `1`) — that is normal. Messages with different keys go to different partitions.
- **Partition 2 shows 0/0/0** — no messages landed on that partition yet. That is **not** an error.


In [2]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          1               1               0               -               -               -
cg-user15-support orders-user15   1          5               5               0               -               -               -
cg-user15-support orders-user15   2          1               1               0               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

### Concept: Leaders, replicas, ISR

From topic `--describe`:

- **Leader** — broker that handles reads/writes for that partition  
- **Replicas** — all copies (RF=3 → three broker ids)  
- **ISR** — copies that are **in sync** with the leader  

**Healthy lab topic:** **Isr** lists the same broker ids as **Replicas** (the order can differ, for example `Replicas: 2,1,3` and `Isr: 2,3,1`).

If **Isr is smaller than Replicas**, the partition is **under-replicated**. Producers with `acks=all` and consumers can then hit timeouts or lag even when some brokers look fine.

**Your run:** three partitions, RF=3, full ISR on all — **brokers look healthy** for your live topic.


In [3]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Topic: orders-user15	TopicId: u5fx7o-2RfW2LvjgNRKbPw	PartitionCount: 3	ReplicationFactor: 3	Configs: min.insync.replicas=2,message.format.version=3.0-IV1,unclean.leader.election.enable=true
	Topic: orders-user15	Partition: 0	Leader: 2	Replicas: 2,1,3	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 1	Leader: 1	Replicas: 1,3,2	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 2	Leader: 3	Replicas: 3,2,1	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

## 2 — Sample application logs

### Purpose of these two files

These are **sample incident logs for practice** — they are **not** from your live cluster. They show what a **Java order application** logged during a bad time window (about **10:14–10:26 UTC on 2026-08-20**). Use them to learn how to read tickets.

| File | Simulates |
|------|------------|
| `producer-error.log` | App **sending** orders to Kafka — fails |
| `consumer-error.log` | App **reading** orders — kicked out of its group |

### How to read any log line

```
TIMESTAMP   LEVEL   LOGGER-NAME   -   MESSAGE
```

| LEVEL | Meaning |
|-------|--------|
| INFO | Normal startup / config |
| WARN | Problem building — read what comes next |
| ERROR | Failure — what the ticket reports |

**Support habit:** find the **first WARN/ERROR**, then read **lines before it** for root cause. Final `TimeoutException` is often a **symptom**, not the root cause.


### Producer log — timeline (what to look for)

| Time | Line type | Meaning |
|------|-----------|--------|
| 10:14 | INFO `ProducerConfig` | **`acks=all`** — wait for all in-sync replicas; **`retries=5`** |
| 10:18:41 | WARN `NetworkClient` | Cannot connect — network, SG, or broker down |
| 10:18:46 | WARN `NOT_LEADER_OR_FOLLOWER` | Leader moved — client **retries** automatically |
| 10:20:42 | ERROR `TimeoutException` | **`delivery.timeout.ms=120000`** expired — message lost from app view |

**Acks reminder:** `acks=0` (fire-and-forget), `acks=1` (leader only), `acks=all` (safest — matches this log).

**One-line story:** broker unreachable → leader wrong → retries exhausted → publish timeout.

**For assignment:** exception type, topic, timestamp, retry lines before final ERROR.


### Sample producer log

The cell below prints a recorded producer log from a real-shaped incident. It is a file on disk, so you can read it as often as you like.

### How to read any Kafka application log

Every line has the same four parts:

| Part | Example | What it gives you |
|------|---------|-------------------|
| Timestamp | `2026-08-20 10:14:02,331` | The time, to the millisecond. Your correlation key for everything else. |
| Level | `INFO`, `WARN`, `ERROR` | How serious the line is |
| Logger | `org.apache.kafka.clients.NetworkClient` | Which component wrote it — often the fastest clue |
| Message | `Connection to node 1 could not be established` | What actually happened |

**Read the logger name first.** It narrows the problem before you have understood the message:

| Logger | What it deals with |
|--------|--------------------|
| `NetworkClient` | Connections to brokers — a connectivity problem |
| `Sender` | The producer's send path, batching and retries |
| `ConsumerCoordinator` | Group membership, rebalancing, offset commits |
| `com.example.orders.*` | The application's own code, not Kafka |

**Read the file in time order, not error-first.** The loudest `ERROR` is usually the last consequence, several minutes after the `WARN` that explains it.


In [1]:
%%cmd
type ..\day-02\samples\producer-error.log

2026-08-20 10:14:02,101 INFO  org.apache.kafka.clients.producer.ProducerConfig - ProducerConfig values:
	acks = all
	bootstrap.servers = [b-1-public.example.kafka.amazonaws.com:9196]
	delivery.timeout.ms = 120000
	enable.idempotence = false
	retries = 5
	retry.backoff.ms = 100
	request.timeout.ms = 30000

2026-08-20 10:14:03,220 INFO  org.apache.kafka.clients.Metadata - [Producer clientId=order-producer-lab] Cluster ID: msk-lab-demo

2026-08-20 10:18:41,004 WARN  org.apache.kafka.clients.NetworkClient - [Producer clientId=order-producer-lab] Connection to node 1 (b-1-public.example.kafka.amazonaws.com/203.0.113.10:9196) could not be established. Broker may not be available.

2026-08-20 10:18:41,188 INFO  org.apache.kafka.clients.NetworkClient - [Producer clientId=order-producer-lab] Give up sending metadata request since no node is available

2026-08-20 10:18:46,201 WARN  org.apache.kafka.clients.producer.internals.Sender - [Producer clientId=order-producer-lab] Got error produce respo

### The producer settings in the log header — what each one means

The first block of the log is the producer printing its own configuration at startup. On a real ticket this is gold: it tells you exactly how the client was set up, without asking the application team.

| Setting | Value here | What it means |
|---------|-----------|---------------|
| `bootstrap.servers` | `b-1-public...:9196` | The first broker the client contacts. It is only a starting point — once connected, the client learns the addresses of every broker in the cluster. |
| `acks` | `all` | How many replicas must confirm a write before it counts as successful. `all` waits for the leader **and** every in-sync replica. Safest, but slowest and most sensitive to replica problems. |
| `retries` | `5` | How many times the client re-sends a message after a **retriable** error, such as the leader moving to another broker. |
| `retry.backoff.ms` | `100` | Wait 100 milliseconds between those retries, so a struggling broker is not flooded. |
| `request.timeout.ms` | `30000` | How long to wait for a reply to **one** network request (30 seconds) before treating that attempt as failed. |
| `delivery.timeout.ms` | `120000` | The **total** time budget for a message — from `send()` until it either succeeds or fails permanently. Two minutes, covering all retries. |
| `enable.idempotence` | `false` | Idempotence is **off**, so a retry can create a **duplicate** message. With `true`, Kafka detects and discards a retried duplicate. |

### How these settings produced the failure

Read them together and the failure explains itself:

1. `acks=all` means every send waits for all in-sync replicas to confirm.
2. The broker became unreachable, which is a **retriable** error, so the client retried — up to `retries=5`, waiting `retry.backoff.ms=100` between attempts.
3. The retries kept failing. After `delivery.timeout.ms=120000` (2 minutes) the message ran out of time and was discarded.
4. That produced the final error: `Expiring 1 record(s) ... 120000 ms has passed since batch creation`.

So the `TimeoutException` is not the root cause. It is the **stopwatch running out** while the real problem — an unreachable broker — went unfixed.


### Producer log vocabulary

These words appear in almost every producer log. Learn them once and the log stops looking cryptic.

| Term in the log | What it means |
|-----------------|---------------|
| `clientId=order-producer-lab` | A name the application gives itself. It is only a label, but it lets you match this app to entries in broker logs and metrics. |
| `correlation id 184` | A sequence number Kafka attaches to every request so a reply can be matched to the request that caused it. Rising correlation ids carrying the **same** error mean the client is retrying the same work over and over. |
| `batch` | For efficiency the producer groups several messages into one **batch** before sending. The message `Expiring 1 record(s) for orders-lab-0:120000 ms has passed since batch creation` means that batch sat in the send buffer for two minutes and was then thrown away. |
| `orders-lab-0` | Topic **and** partition together: partition `0` of topic `orders-lab`. |
| `node 1` / `(id: 1 rack: null)` | The broker's id number. `rack` is an optional label describing which rack or availability zone a broker sits in, used for spreading replicas. `null` simply means no rack label is configured — it is **not** an error. |
| `NOT_LEADER_OR_FOLLOWER` | The client sent a write to a broker that is no longer the leader for that partition. The leader moved. The client refreshes its cluster metadata and retries automatically, so a few of these lines are normal during a leader change. |
| `Broker may not be available` | The client could not open a network connection to that broker at all — a connectivity problem, not an authorization or data problem. |


### Mine the producer log (`findstr`)

Reading a whole log line by line is slow. `findstr` is the Windows equivalent of `grep` — it prints only the lines that contain the words you ask for.

The next cell searches the producer log for the words that matter in an incident: `WARN`, `ERROR`, `TimeoutException`, `NOT_LEADER`, `retry`, and `acks`. The `/i` switch makes the match case-insensitive.

This is exactly how you would quickly pull the important lines out of a large production log file before writing a ticket.


In [ ]:
%%cmd
findstr /i "WARN ERROR TimeoutException NOT_LEADER retry acks" ..\day-02\samples\producer-error.log


### Consumer log — timeline (what to look for)

| Time | Line type | Meaning |
|------|-----------|--------|
| 10:19:11 | INFO joined group | Healthy — partitions 0,1,2 assigned (**rebalance** finished) |
| 10:24:12 | WARN **poll timeout expired** | App blocked longer than **`max.poll.interval.ms`** — group coordinator will revoke partitions |
| 10:24:12 | ERROR **Offset commit failed** | Consumer **removed from group** — progress not saved |
| 10:26:40+ | ERROR **DisconnectException** | Consumer loop stopped |

**Rebalancing:** when a member is slow or leaves, Kafka **reassigns partitions** to remaining consumers — brief pause in processing.

**Day 2 lesson:** producer and consumer failures look **different**. Always check **both logs** plus `kafka-consumer-groups --describe`.

**For assignment:** group id, poll timeout, commit failed, disconnect.


### Sample consumer log

Now the same incident from the reading side. The consumer log uses a different vocabulary from the producer log, because it is concerned with **group membership** rather than sending.

### What to look for, in order

| Stage | What it looks like | What it tells you |
|-------|--------------------|-------------------|
| Startup | The consumer prints its configuration | How this client was actually set up |
| Discovery | `Discovered group coordinator` | It found the broker that manages its group |
| Join | `Successfully joined group`, partitions assigned | Healthy — it is now allowed to read |
| Trouble | `poll timeout has expired` | Processing took too long between `poll()` calls |
| Eviction | Partitions revoked | The group took its work away |
| Failure | `Offset commit failed` | Its progress was never saved |

That sequence is the classic **slow consumer**. Nothing was wrong with the broker; the application simply could not process a batch fast enough, so the group removed it and reassigned its partitions.

**The consequence to carry into the ticket:** because the final commit failed, the messages processed since the last successful commit are **not recorded**. When the application restarts it will process them again. Somebody downstream needs to know that.


In [1]:
%%cmd
type ..\day-02\samples\consumer-error.log

2026-08-20 10:19:10,044 INFO  org.apache.kafka.clients.consumer.ConsumerConfig - ConsumerConfig values:
	bootstrap.servers = [b-1-public.example.kafka.amazonaws.com:9196]
	group.id = cg-lab-support
	enable.auto.commit = true
	max.poll.interval.ms = 300000
	session.timeout.ms = 45000
	auto.offset.reset = earliest

2026-08-20 10:19:11,201 INFO  org.apache.kafka.clients.consumer.internals.ConsumerCoordinator - [Consumer clientId=order-consumer-lab, groupId=cg-lab-support] Discovered group coordinator b-2-public.example.kafka.amazonaws.com:9196 (id: 2147483646 rack: null)

2026-08-20 10:19:11,388 INFO  org.apache.kafka.clients.consumer.internals.ConsumerCoordinator - [Consumer clientId=order-consumer-lab, groupId=cg-lab-support] Successfully joined group with generation Generation{generationId=7, memberId='order-consumer-lab-1', protocol='range'}

2026-08-20 10:19:11,401 INFO  org.apache.kafka.clients.consumer.internals.ConsumerCoordinator - [Consumer clientId=order-consumer-lab, groupId=c

### The consumer settings in the log header — what each one means

Just like the producer, the consumer prints its configuration when it starts.

| Setting | Value here | What it means |
|---------|-----------|---------------|
| `bootstrap.servers` | `b-1-public...:9196` | The first broker contacted, used to discover the rest of the cluster. |
| `group.id` | `cg-lab-support` | The **consumer group** this application belongs to. Every consumer sharing this id shares the work: partitions are split between them and they share one set of committed offsets. |
| `enable.auto.commit` | `true` | The client saves its progress automatically in the background instead of the application deciding when to commit. Convenient, but if the app dies between commits the same messages get processed twice. |
| `max.poll.interval.ms` | `300000` | The longest gap allowed **between two `poll()` calls** — 5 minutes. If the application takes longer than this to process a batch, Kafka assumes it is stuck and removes it from the group. **This is the setting that was breached in this log.** |
| `session.timeout.ms` | `45000` | How long the broker waits for a **heartbeat** before declaring the consumer dead — 45 seconds. Heartbeats are sent on a background thread, separate from message processing. |
| `auto.offset.reset` | `earliest` | What to do when the group has **no** saved offset at all: `earliest` starts at the oldest message still retained; `latest` would start only with new messages. |

### Why two different timeouts?

This confuses people, so it is worth being clear:

- **`session.timeout.ms`** watches the **heartbeat** thread. It answers: *is this process still alive?*
- **`max.poll.interval.ms`** watches the **processing** loop. It answers: *is this process still doing useful work?*

An application can be perfectly alive (heartbeats fine) while being stuck processing one huge batch. That is exactly what happened here: heartbeats kept flowing, but `poll()` was not called again in time, so the group removed it anyway.


### Consumer log vocabulary — coordinator, rebalance, offset commit

The consumer log talks about **group membership**, so it uses a different vocabulary from the producer log.

| Term | What it means |
|------|---------------|
| **Coordinator** | A job that one broker performs on behalf of a consumer group. |
| **Group coordinator** | The specific broker chosen to manage `cg-lab-support`. It tracks which members are alive, decides who reads which partition, and stores the group's committed offsets. |
| **`ConsumerCoordinator`** | The class **inside the Kafka client library** that talks to the group coordinator. When you see this name in a log line, the message is about joining the group, rebalancing, or committing offsets — never about the message content itself. |
| **`clientId=order-consumer-lab`** | A label the application gives itself, the same idea as the producer's clientId. |
| **`groupId=cg-lab-support`** | The consumer group this member joined. |
| **Generation** | A version number for the group's membership. Every rebalance increases it by one, so `generationId=7` means the group has re-formed seven times. |
| **Rebalance** | Kafka re-dividing the topic's partitions among whichever members are currently alive. Processing pauses briefly while this happens. |
| **Revoke partitions** | Taking partitions **away** from a member. When the coordinator decides a member is stuck or gone, it revokes that member's partitions and gives them to others. The revoked member is no longer allowed to process or commit for those partitions. |
| **Offset commit failed** | The consumer tried to record "I have processed up to offset N" and was refused, because it is **no longer a member** of the group — its partitions were already revoked. The work it just did was **not** recorded, so after a restart those same messages will be delivered again. |

### Reading one line end to end

Take this line from the log:

```
2026-08-20 10:19:11,201 INFO  org.apache.kafka.clients.consumer.internals.ConsumerCoordinator - [Consumer clientId=order-consumer-lab, groupId=cg-lab-support] Discovered group coordinator b-2-public.example.kafka.amazonaws.com:9196 (id: 2147483646 rack: null)
```

Broken into pieces:

| Part | Meaning |
|------|---------|
| `2026-08-20 10:19:11,201` | Date and time, down to milliseconds |
| `INFO` | Severity — normal progress, nothing wrong |
| `org.apache...ConsumerCoordinator` | Which part of the client wrote the line: group membership logic |
| `clientId=order-consumer-lab` | The application's own label |
| `groupId=cg-lab-support` | The consumer group it belongs to |
| `Discovered group coordinator` | It has found the broker that manages this group |
| `b-2-public...:9196` | That broker's address |
| `id: 2147483646` | The coordinator's internal node id. Very large numbers are normal here: Kafka calculates it as `2147483647 minus the broker id`, so a coordinator id can never collide with a real broker id. `2147483646` therefore means **broker 1**. |
| `rack: null` | No rack or availability-zone label is configured. Not an error. |

**In plain words:** *at 10:19:11 the consumer application `order-consumer-lab`, a member of group `cg-lab-support`, located the broker that manages its group. This is a normal startup step.*

### The failure chain in this log

Now the sequence makes sense:

1. **Joined group** — healthy, partitions 0, 1 and 2 assigned.
2. **`poll timeout has expired`** — the app did not call `poll()` within `max.poll.interval.ms`.
3. The coordinator **revoked its partitions** and removed it from the group.
4. **`Offset commit failed`** — the now-evicted member tried to save progress and was refused.
5. **Disconnect** — the consumer loop stopped.

The business impact: messages between the last successful commit and the failure will be **processed a second time** when the app restarts.


### Mine the consumer log with `findstr`

Reading a whole log works for a 15-line sample. A real log is tens of thousands of lines, so you filter it.

`findstr` is the Windows equivalent of `grep`. The flags used here:

| Flag | Meaning |
|------|---------|
| `/i` | Case-insensitive, so `ERROR` and `error` both match |
| `/c:"text"` | Match this exact phrase, spaces included, rather than each word separately |

### What to search for, and why these terms

| Search term | What it finds |
|-------------|---------------|
| `CommitFailedException` | The consumer lost its group membership before it could save progress |
| `poll timeout` | Processing exceeded `max.poll.interval.ms` — the root of the problem |
| `Revoke` or `revoked` | The moment the group took partitions away |
| `coordinator` | Every group-membership event, which gives you the rebalance timeline |

**The technique to take away:** search for the **error class name** first, because it is unique and unambiguous. Then search for the surrounding words to get the timeline around it. Two searches usually locate an incident in a log you have never seen before.


In [ ]:
%%cmd
findstr /i "WARN ERROR poll timeout CommitFailed rebalance Disconnect" ..\day-02\samples\consumer-error.log


### Reading — line up the log times with broker metrics

The sample logs cover roughly **10:14 to 10:26 UTC on 2026-08-20**. That timestamp is written inside these recorded files — it is not today's date, so do not go looking for it on your live cluster.

The **technique** is what matters, and you will reuse it on every incident:

1. Take the **first** and **last** error timestamp from the application log.
2. Look at the **broker metrics for that same window**.
3. Ask one question: *while the application was failing, were the brokers busy, full, or idle?*

If the brokers were idle, the problem is on the application side or in the network path. If they were pinned at high CPU or nearly out of disk, the broker is a genuine suspect.

### Two different things in CloudWatch

Keep these apart — they answer different questions:

| CloudWatch feature | What it holds | Question it answers |
|--------------------|---------------|---------------------|
| **CloudWatch Metrics** | Numeric time series published by MSK, such as `CpuIdle`, `KafkaDataLogsDiskUsed` and `BytesInPerSec` | *Was the broker under stress at that moment?* |
| **CloudWatch Logs** | Actual broker log lines, delivered only if broker log delivery is switched on for the cluster | *What did the broker itself say at that moment?* |

The producer and consumer logs you just read are **neither** of these. They are files written by the Java application on its own host, which is why they can show timeouts even when the cluster is perfectly healthy.

You will work with both CloudWatch Metrics and CloudWatch Logs on **Day 3**, against a live window with traffic that you generate yourself.


## 3 — Broker health

### Concept: Broker health checklist

Cross-check logs against infrastructure:

1. **Cluster Active?** — `aws kafka describe-cluster`  
2. **Partitions healthy?** — Replicas vs ISR on topic describe  
3. **Can this PC reach brokers?** — `nslookup` on bootstrap hostname (strip `:9196` from `%BOOTSTRAP%`)

If **Kafka CLI works** from your lab PC but **application logs** show timeouts, the problem is often on the **application host** (wrong bootstrap, firewall, old config). The whole MSK cluster is usually **not** fully down.

**What you should see:**

- AWS cluster state **ACTIVE** (if your CLI account is correct).
- Topic `--describe`: **Isr** equals **Replicas** on each partition.
- **nslookup:** lines with **Name:** and **Address(es)** for the public bootstrap host.

**If AWS returns AccessDenied:** your CLI user is in the wrong account (see output — lab cluster is account `891377046325`). For the assignment write: *“Kafka CLI healthy; AWS describe-cluster AccessDenied — cannot confirm ACTIVE from CLI.”* Kafka + ISR checks still count.


### Confirm the bootstrap strings

Before drawing any conclusion about broker health, confirm your client is aimed at the right place. A misdirected client produces symptoms identical to a broken cluster.

### What you are checking for

| Requirement | Why |
|-------------|-----|
| The hostname contains **`-public`** | Your machine is outside the cluster's VPC, so only the public listener is reachable |
| The port is **`9196`** | That is the public SCRAM listener. 9198 is public IAM; 9096 and 9098 are VPC-internal. |

### Why this check comes first

From outside the VPC, the private `:9096` endpoint does not return a helpful error. It hangs, then times out. So a wrong bootstrap string and a genuinely unreachable broker look exactly the same from the client.

Thirty seconds spent confirming the endpoint saves you from investigating broker capacity for a problem that is really a single line of client configuration. On Day 5 you will meet a ticket where this is the whole answer.


In [1]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
aws kafka get-bootstrap-brokers --cluster-arn %CLUSTER_ARN% --region %REGION%


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>aws kafka get-bootstrap-brokers --cluster-arn %CLUSTER_ARN% --region %REGION%



aws: [ERROR]: An error occurred (AccessDeniedException) when calling the GetBootstrapBrokers operation: User: arn:aws:iam::410232017221:user/u1 is not authorized to perform: kafka:GetBootstrapBrokers on resource: arn:aws:kafka:ap-south-1:891377046325:cluster/msk-kafka-class/ad474b96-f594-495b-82cd-ad95d7c2c71c-4 because no resource-based policy allows the kafka:GetBootstrapBrokers action



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

### Concept: Network path (Security Groups)

Your Windows lab PC connects to MSK **from outside the VPC** on public SCRAM port **9196**.

If **DNS works** but Kafka **hangs or times out**, common causes are:

1. Wrong bootstrap (private hostname or wrong port).
2. **Security Group** on MSK not allowing your client IP on 9196 / 9198.

You do **not** change Security Group rules in this lab. Collect the bootstrap string, the nslookup result, and the exact Kafka error, then **escalate to the network or cloud team** with that evidence.

**Check order:** DNS → bootstrap is `-public` and `:9196` → SCRAM works → then suspect SG with infra.


In [6]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
aws kafka describe-cluster --cluster-arn %CLUSTER_ARN% --region %REGION% --query "ClusterInfo.{State:State,Brokers:NumberOfBrokerNodes}" --output table

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>aws kafka describe-cluster --cluster-arn %CLUSTER_ARN% --region %REGION% --query "ClusterInfo.{State:State,Brokers:NumberOfBrokerNodes}" --output table



aws: [ERROR]: An error occurred (AccessDeniedException) when calling the DescribeCluster operation: User: arn:aws:iam::410232017221:user/u1 is not authorized to perform: kafka:DescribeCluster on resource: arn:aws:kafka:ap-south-1:891377046325:cluster/msk-kafka-class/ad474b96-f594-495b-82cd-ad95d7c2c71c-4 because no resource-based policy allows the kafka:DescribeCluster action



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

### Reading §3.1 output

| Result | Meaning |
|--------|--------|
| **AccessDenied** on `describe-cluster` | AWS CLI user not authorized for this MSK cluster — fix `aws configure` for your lab account, or note it in the assignment |
| **Kafka `--describe` works + full ISR** | Data plane is fine; sample **producer/consumer logs** are a separate fictional incident |
| **nslookup (next cell)** | Confirms your PC can resolve the public broker hostname |


In [7]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Topic: orders-user15	TopicId: u5fx7o-2RfW2LvjgNRKbPw	PartitionCount: 3	ReplicationFactor: 3	Configs: min.insync.replicas=2,message.format.version=3.0-IV1,unclean.leader.election.enable=true
	Topic: orders-user15	Partition: 0	Leader: 2	Replicas: 2,1,3	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 1	Leader: 1	Replicas: 1,3,2	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 2	Leader: 3	Replicas: 3,2,1	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

In [1]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
for /f "tokens=1 delims=:" %%a in ("%BOOTSTRAP%") do set BROKER_HOST=%%a
echo Resolving %BROKER_HOST%
nslookup %BROKER_HOST%

Resolving b-1-public.mskkafkaclass.qau5zr.c4.kafka.ap-south-1.amazonaws.com
Server:  UnKnown
Address:  2401:9640:2f31::6

Name:    b-1-public.mskkafkaclass.qau5zr.c4.kafka.ap-south-1.amazonaws.com
Address:  43.204.58.22

Non-authoritative answer:



### Reading — nslookup

`nslookup` asks only one question: **does this hostname resolve to an IP address?** It does not test whether the port is open, whether TLS works, or whether your credentials are valid.

That narrowness is what makes it useful. It separates two failures that look identical to a user:

| Result | What it means | Who owns the next step |
|--------|---------------|------------------------|
| One or more **Address** lines | DNS is fine. If the connection still fails, the problem is the port, a Security Group, or authentication. | The network team, with your evidence |
| **Non-existent domain** | The hostname is wrong, or the endpoint changed after a cluster update | Whoever configured the client |
| **Request timed out** | The DNS resolver itself is unreachable from this machine | The network team |

**What to do with a successful lookup.** DNS resolving proves the name is valid, and it makes the remaining possibilities much shorter: a blocked port, a Security Group that does not allow your source address, or wrong credentials.

You do **not** change Security Group rules in this lab. Collect three things — the bootstrap string, the nslookup output, and the exact Kafka error — and escalate with all three. An escalation carrying that evidence gets actioned; one saying "Kafka is not working" gets sent back for details.


### Concept: Under-replicated partitions (when ISR < Replicas)

Every partition has a list of **Replicas** (all the copies) and an **ISR** — the In-Sync Replicas that are fully caught up with the leader. When a replica falls behind, or its broker goes down, it drops out of the ISR and the partition becomes **under-replicated (URP)**.

Your **live lab cluster** is healthy, so it usually shows **full ISR** (ISR = Replicas). To practise reading a URP the way it appears on a real ticket, look at the **saved example output** in the next cell. It is a recorded example, not your live topic.


In [1]:
%%cmd
type ..\day-02\samples\topic-describe-urp-snippet.txt


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>type ..\day-02\samples\topic-describe-urp-snippet.txt
# Sample output from: kafka-topics --describe (fictional incident — NOT your live topic)
# Use this when your live cluster shows full ISR, but a ticket mentions under-replication.

Topic: orders-lab	TopicId: demo-urp-example	PartitionCount: 3	ReplicationFactor: 3	Configs: min.insync.replicas=2
	Topic: orders-lab	Partition: 0	Leader: 2	Replicas: 2,1,3	Isr: 2,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-lab	Partition: 1	Leader: 1	Replicas: 1,3,2	Isr: 1,3,2	Elr: N/A	LastKnownElr: N/A
	Topic: orders-lab	Partition: 2	Leader: 3	Replicas: 3,2,1	Isr: 3,2,1	Elr: N/A	LastKnownElr: N/A

# How to read partition 0:
# - Replicas: 2,1,3 means Kafka wants a copy on brokers 2, 1, and 3.
# - Isr: 2,1 means only brokers 2 and 1 are in sync. Broker 3 is missing = UNDER-REPLICATED.
# - With acks=all, produce to

### Reading — under-replicated partition (sample)

Look at **partition 0** in the saved example output above:

- **Replicas: 2,1,3** — Kafka expects a copy of the data on brokers 2, 1, and 3.
- **Isr: 2,1** — only brokers 2 and 1 are **in sync**. Broker **3** is missing from the in-sync list.

This is an **under-replicated partition (URP)**. The copy on broker 3 is behind or offline.

**Why it matters:** with `acks=all`, the producer waits for all in-sync replicas. If one replica is out of sync, produce can **slow down or fail** even when the cluster looks healthy.

**On a ticket:** fix broker or replica health (disk, CPU, network). **Do not** only restart the consumer — that does not fix missing replicas.

**Your live cluster:** run topic `--describe` on `%TOPIC%`. If **Isr** matches **Replicas** on every partition, write **ISR healthy**. The snippet is practice for tickets that mention URP when your lab cluster is healthy.


## 4 — Recover a stopped consumer

### The situation

This is one of the most common Kafka tickets you will ever receive:

> "Orders stopped appearing in the dashboard after 09:40. Nothing else was changed."

In most cases **nothing is wrong with Kafka**. The consumer application stopped — it crashed, was redeployed, lost its network, or was shut down and never restarted. Producers carried on writing, so messages piled up on the topic and **LAG grew**. The cluster is healthy; the reader is missing.

### What "recover" actually means

Recovery is not "restart it and hope". It is three deliberate steps, and each one produces evidence you can paste straight into the ticket:

| Step | What you do | Evidence it gives you |
|------|-------------|-----------------------|
| **1. Measure before** | `--describe` the group and write down LAG for each partition | Proof of how far behind it was when you picked up the ticket |
| **2. Prove the path** | Send one test message, then start the consumer | Proof that a **brand new** message can travel producer → topic → consumer **right now** |
| **3. Measure after** | `--describe` again | Proof that the backlog drained and LAG returned to 0 |

### What a "test message" is

A **test message** — often called a **probe** — is an ordinary Kafka message whose text you choose yourself so that you can recognise it instantly in the consumer output. There is nothing special about it as far as Kafka is concerned; it is a normal record on your topic.

In this lab the text is **`recover-probe`**. Any string would do. The value is in the timing: when you see that exact word come out of the consumer, you know the message you **just sent, seconds ago** travelled the entire path successfully. You are not being fooled by old data that happened to be sitting on the topic.

This is much stronger evidence than "the process is running again", which only tells you something started, not that it works.

### How this lab differs from a real server

On a real server you would leave the consumer running in one window and produce from a second window, watching messages arrive live.

A notebook cell cannot stay open like that, so here you **produce first, then run the consumer** with a message limit and a timeout. The outcome is identical; only the order on screen changes.


In [9]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo === BEFORE recover ===
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>echo === BEFORE recover ===
=== BEFORE recover ===

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          1               1               0               -               -               -
cg-user15-support orders-user15   1          5               5               0               -               -               -
cg-user15-support orders-user15   2          1               1               0               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

### The command you are about to run

`call ..\scripts\produce.bat recover-probe`

| Part | Meaning |
|------|---------|
| `call` | Run another batch file, then come back here when it finishes |
| `..\scripts\produce.bat` | The produce helper — from the `day-02` folder, go up one level, then into `scripts` |
| `recover-probe` | The text of the message being sent |

The helper connects to `%BOOTSTRAP%` using your SCRAM credentials, writes the single line `recover-probe` to `%TOPIC%`, and exits.

**What you should see:** only `log4j WARN` lines, then the prompt returns. The producer does not echo your message back, so a quiet return is the sign that the send succeeded.

Outside Jupyter, in a normal Command Prompt, the same command is:

`call %USERPROFILE%\Apache-Kafka-on-AWS\scripts\produce.bat recover-probe`


In [10]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\produce.bat recover-probe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\produce.bat recover-probe


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

### Now read the message back

`call ..\scripts\consume.bat --from-beginning --max-messages 15 --timeout-ms 25000`

| Flag | Why it is there |
|------|-----------------|
| `--from-beginning` | Start from the oldest retained message **only if this group has no saved position**. If your group already has committed offsets — and after the earlier sections it does — Kafka uses those instead, so you see only new messages. |
| `--max-messages 15` | Stop after 15 messages so the cell cannot run forever |
| `--timeout-ms 25000` | Give up after 25 seconds if nothing more arrives |

**What you should see:** the line `recover-probe`, followed by `Processed a total of N messages`.

Very often `recover-probe` is the **only** line printed. That is a success, not a problem: your group already committed its position earlier, so there is no backlog to replay and the probe is genuinely the only new message.


In [11]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\consume.bat --from-beginning --max-messages 15 --timeout-ms 25000

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\consume.bat --from-beginning --max-messages 15 --timeout-ms 25000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.consumer.ConsumerConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


recover-probe


Processed a total of 1 messages



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

In [12]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo === AFTER recover — LAG should be 0 or lower ===
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>echo === AFTER recover — LAG should be 0 or lower ===
=== AFTER recover — LAG should be 0 or lower ===

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          1               1               0               -               -               -
cg-user15-support orders-user15   1          5               5               0               -               -               -
cg-user15-support orders-user15   2          2               2               0               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-02>

### Reading — before versus after

Put the two `--describe` outputs side by side and compare four things:

| What to compare | Healthy recovery looks like |
|-----------------|------------------------------|
| **LAG** on each partition | Back to **0** everywhere |
| **CURRENT-OFFSET** | Moved forward on the partition that received the probe, for example `1` → `2` |
| **LOG-END-OFFSET** | Also moved forward by one, because you added exactly one message |
| Consumer output | Contains the line `recover-probe` |

**If LAG is still above 0:** the consumer simply did not run long enough to drain the backlog. Increase `--max-messages`, or run the consume cell again — each run continues from the saved position, so nothing is lost.

**If `recover-probe` never appears:** check that the produce cell and the consume cell are pointing at the same `%TOPIC%`, and that the produce cell finished without an error.

### What you would write on the ticket

> Consumer group `cg-userN-support` showed LAG of X on partition P at HH:MM. Restarted the consumer path and published a test message. The test message was consumed successfully and LAG returned to 0 on all partitions. Message flow is restored. No offsets were reset and no data was lost.


### Worksheet — lag before / after recover

| Partition | LAG before | LAG after | Saw `recover-probe`? |
|-----------|------------|-----------|----------------------|
| 0 | | | |
| 1 | | | |
| 2 | | | |

Fill the table from your `--describe` output before and after recover. **Healthy** means LAG 0 and you saw **`recover-probe`** in consume output.


### Optional — deliberately wrong bootstrap (expect failure)

This cell is designed to fail, and the failure is the lesson.

It points the Kafka CLI at `127.0.0.1:9196` — your own machine, where no Kafka broker is running. Everything else about the command is correct: the same `%CLIENT%` file, the same valid SCRAM username and password, the same flags.

**The point of the exercise.** Only the address is wrong, so whatever error comes back must be a *connectivity* error rather than a credentials error. Once you have seen the difference once, you can tell the two apart instantly on a real ticket.

**Do not save a wrong bootstrap value** into your session file. This cell hard-codes the bad address so your real `%BOOTSTRAP%` stays intact.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
type ..\day-02\samples\wrong-bootstrap-expect.txt
echo Expect fast failure — nothing listening on localhost:
kafka-topics.bat --bootstrap-server 127.0.0.1:9196 --command-config %CLIENT% --list


### Reading — connectivity errors versus authentication errors

**What you should see:** a connection or timeout error, within roughly 30 seconds. Nothing is listening on `localhost:9196`, so the client never gets far enough to attempt authentication.

### Telling the two failure types apart

| The error mentions | The failure is | What to check |
|--------------------|----------------|---------------|
| `Connection to node -1 could not be established`, `Broker may not be available`, or a plain timeout | **Connectivity** — you never reached a broker | The hostname, the port, DNS, Security Groups |
| `Authentication failed`, `SaslAuthenticationException` | **Credentials** — you reached the broker and it rejected you | The username and password in `%CLIENT%` |
| `TopicAuthorizationException` | **Permissions** — you were authenticated, then refused | The ACLs on that topic (Day 4) |

**Why this ordering matters.** These three point at three different teams. Reading the error name before doing anything else is the single fastest way to avoid investigating the wrong layer — and it is the habit that Day 5's scenarios are built around.

### On a real ticket

From a machine outside the cluster's VPC, the bootstrap must be a **`-public`** hostname on port **`9196`** for SCRAM. The private `:9096` endpoint does not produce a helpful message from outside; it simply hangs and then times out, which looks identical to a broker being down.


## Assignment

Fill [samples/assignment-incident.md](samples/assignment-incident.md) using:

- **Live cluster:** your `--describe` output (lag, ISR) — e.g. LAG 0, full ISR, AccessDenied on AWS if applicable  
- **Sample logs:** lines from **producer** and **consumer** files (`orders-lab`, not your live topic)  

Keep answers short — one sentence per finding is enough.

**Tip:** Contrast *live* health (LAG 0, ISR full) with the *sample* incident (timeouts, poll expired) — that is the Day 2 story.
